# Feature selection

Separate features based on the classification accuracy. 

NOTE: dependence on choice of *thresholds*.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt


In [ ]:
plt.style.use("my_style.mplstyle")

In [ ]:
models_dsct=['GNO', 'UNO', 'PINK', 'BROWN', 'VIOLET', # noises (R)
        'AR1_GNO', 'STAR_GNO', #    ARMA (R)
        'ARNOLD', 'CHIRIKOV', # Conservative chaotic maps (R)
        'HENR_diverse', 'HENR_same', 'QUADRATIC_RSUM', # Sum of frwd and bckwd realisations of chaotic maps (R)
        'AR1_UNO', 'ARMA11_UNO', 'AR3_Gamma', 'N_AR2', 'SETAR1_GNO', 'SETAR2_GNO',
        'HEN', 'HEN_SUM', 'LOGISTIC4', 'LOGISTIC38284', 'QUADRATIC', # Chaotic maps (I)
        'MODA', 'LLOG', # Other deterministic (I)
        'SINE_STOCH' # Other stochastic (I)
        ]



models_cnt = ['BRW_cont','OU', 'Oscillator', 
              'LORENZ_SUM', 'ROSSLER_SUM', 
              'MACKEYGLASS17', 'VDP', 
              'LORENZ_STOCH_SUM',
                'VDP_STOCH']


models = models_dsct + models_cnt
print('Models:', models)

model_keywords={ # discrete
                'GNO': 'reversible', 'UNO': 'reversible', 'PINK': 'reversible', 'BROWN': 'reversible', 'VIOLET': 'reversible',
                'AR1_GNO': 'reversible', 'STAR_GNO': 'reversible',
                'ARNOLD': 'reversible', 'CHIRIKOV': 'reversible',
                'HENR_diverse': 'reversible', 'HENR_same': 'reversible', 'QUADRATIC_RSUM': 'reversible',
                'AR1_UNO': 'irreversible', 'ARMA11_UNO': 'irreversible', 'AR3_Gamma': 'irreversible', 'N_AR2': 'irreversible', 'SETAR1_GNO': 'irreversible', 'SETAR2_GNO': 'irreversible', 
                'HEN': 'irreversible', 'HEN_SUM': 'irreversible', 'LOGISTIC4': 'irreversible', 'LOGISTIC38284': 'irreversible','QUADRATIC': 'irreversible',
                'MODA': 'irreversible', 'LLOG': 'irreversible',
                'SINE_STOCH': 'irreversible',
                # continuous
                'BRW_cont': 'reversible', 'OU': 'reversible', 'Oscillator': 'reversible',
                'LORENZ_SUM': 'irreversible', 'ROSSLER_SUM': 'irreversible',
                'MACKEYGLASS17': 'irreversible', 'VDP': 'irreversible',
                'LORENZ_STOCH_SUM': 'irreversible',
                'VDP_STOCH': 'irreversible'
        }


In [ ]:
cwd = os.getcwd()

dir_hctsa= cwd+'/data-analysis/'
dir_accuracy= cwd+'/data-analysis/accuracy/'
dir_zero = cwd+ '/data-zero/'
dir_supp = cwd+ '/supplementary-material/'

In [ ]:
# Load full hctsa matrix
df_hctsa=pd.read_csv(dir_hctsa+'df_TS_DataMat_diff.csv')
df_hctsa.set_index(['Model'], inplace=True)

# Add column with reversible/irreversible label
df_type=df_hctsa.index.get_level_values(0).map(model_keywords)
df_hctsa.insert(0,'Type',df_type)

In [ ]:
# Load zero features
df_zero=pd.read_csv(dir_zero+'df_ops_zero.csv')

In [ ]:
# Look for ties in df_hctsa
# Count how many uneque values are in each column
df_hctsa_counts = df_hctsa.nunique()
# No tie if number of unique values is equal to number of rows
df_hctsa_counts = df_hctsa_counts[df_hctsa_counts != df_hctsa.shape[0]]
# Get the names of the columns with ties
tie_columns = df_hctsa_counts.index.tolist()

In [ ]:
# Load accuracy
df_accuracy=pd.read_csv(dir_accuracy+'/df_accuracy_abs_1NN_sorted.csv')

In [ ]:
# Separate features in 3 groups: good, medium, bad
threshold_high=0.72
threshold_low=0.6

df_good=df_accuracy[df_accuracy['Accuracy']>threshold_high]
df_good.to_csv(dir_accuracy+'/df_accuracy_good_1NN.csv', index=False)

df_medium=df_accuracy[(df_accuracy['Accuracy']<=threshold_high) & (df_accuracy['Accuracy']>threshold_low)]
df_medium.to_csv(dir_accuracy+'/df_accuracy_medium_1NN.csv', index=False)

df_bad=df_accuracy[df_accuracy['Accuracy']<=threshold_low]
df_bad.to_csv(dir_accuracy+'/df_accuracy_bad_1NN.csv', index=False)

In [ ]:
# Extract hctsa data of the three groups

## Good
selected_features=df_good['Operation'].values
df_good_hctsa=df_hctsa[selected_features]
# Add column with reversible/irreversible label
df_type=df_good_hctsa.index.get_level_values(0).map(model_keywords)
df_good_hctsa.insert(0,'Type',df_type)
# Save selected features
df_good_hctsa.to_csv(dir_accuracy+'/df_good_hctsa_1NN.csv')

## Medium
selected_features=df_medium['Operation'].values
df_medium_hctsa=df_hctsa[selected_features]
# Add column with reversible/irreversible label
df_type=df_medium_hctsa.index.get_level_values(0).map(model_keywords)
df_medium_hctsa.insert(0,'Type',df_type)
# Save selected features    
df_medium_hctsa.to_csv(dir_accuracy+'/df_medium_hctsa_1NN.csv')

## Bad
selected_features=df_bad['Operation'].values
df_bad_hctsa=df_hctsa[selected_features]
# Add column with reversible/irreversible label
df_type=df_bad_hctsa.index.get_level_values(0).map(model_keywords)
df_bad_hctsa.insert(0,'Type',df_type)
# Save selected features
df_bad_hctsa.to_csv(dir_accuracy+'/df_bad_hctsa_1NN.csv')



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assume df_accuracy is a pandas DataFrame
# and threshold_high is defined
data = df_accuracy['Accuracy']

# Create histogram
fig, ax = plt.subplots(figsize=(10, 7))  # 800x600 pixels (assuming 100 dpi)

# Histogram
n, bins, patches = ax.hist(data, bins=60, color='silver', edgecolor='grey', linewidth=1.2)

# Vertical line at threshold
#ax.axvline(x=threshold_high, color='grey', linewidth=1.5, linestyle='--')

# Axis labels
ax.set_xlabel('Accuracy', fontsize=24, color='black')
ax.set_ylabel('Number of features', fontsize=24, color='black')

# Tick labels
ax.tick_params(axis='x', labelsize=24, colors='black')
ax.tick_params(axis='y', labelsize=24, colors='black')

# Grid lines
ax.xaxis.grid(True, color='whitesmoke')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.grid()
# no left y axis
ax.spines['left'].set_visible(True)

# Optional: tighten layout
plt.tight_layout()
# Save in svg format
fig.savefig(dir_hctsa+'figures/histogram_accuracy_h.svg', format='svg', dpi=300)

plt.show()


In [ ]:
# Remove zero features from those in the accuracy
df_accuracy_no_zero=df_accuracy[~df_accuracy['Operation'].isin(df_zero['Name'].values)]
# Accuracy of zero features
df_zero_accuracy=df_accuracy[df_accuracy['Operation'].isin(df_zero['Name'].values)]
# Plot histogram of accuracy
data = df_accuracy_no_zero['Accuracy']*100
fig, ax = plt.subplots(figsize=(9, 8))  # 800x600 pixels (assuming 100 dpi)

# Histogram
n, bins, patches = ax.hist(data, bins=40, color='silver', edgecolor='grey', linewidth=1.25)

# Vertical line at threshold
ax.axvline(x=threshold_high*100, color='grey', linewidth=1.5, linestyle=(0, (8, 8)))

# Axis labels
ax.set_xlabel('Accuracy (%)', fontsize=34, color='black')
ax.set_ylabel('Number of features', fontsize=34, color='black')

# Tick labels
ax.tick_params(axis='x', labelsize=32, colors='black')
ax.tick_params(axis='y', labelsize=32, colors='black')

# Grid lines
ax.xaxis.grid(True, color='whitesmoke')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.grid()
# no left y axis
ax.spines['left'].set_visible(True)
#plt.title('Accuracy of features without zero features', fontsize=24, color='black')


# Optional: tighten layout
plt.tight_layout()
# Save in svg format
fig.savefig(dir_hctsa+'figures/histogram_accuracy_no_zero_h.svg', format='svg', dpi=300)

plt.show()

#fig.show()

In [ ]:
# Write to Excel with formatting

# Create the save directory if it doesn't exist
os.makedirs(dir_supp, exist_ok=True)


with pd.ExcelWriter(dir_supp+'Supplement_Table_S2.xlsx', engine='xlsxwriter') as writer:
    df_accuracy_no_zero.to_excel(writer, index=False, sheet_name='Sheet1')

    workbook  = writer.book
    worksheet = writer.sheets['Sheet1']

    # Define formats
    format_grey = workbook.add_format({'bg_color': '#DDDDDD'})
    format_white = workbook.add_format({'bg_color': '#FFFFFF'})

    # Specify the header names in bold
    bold_format = workbook.add_format({'bold': True, 'font_color': '#000000', 'bg_color': '#DDDDDD'})
    header_names = ['Name', 'Global average', 'Feature class', 'Accuracy']
    # Write the header with bold format
    for col_num, header in enumerate(df_accuracy_no_zero.columns):
        worksheet.write(0, col_num, header, bold_format)


    # Apply alternating colors row by row
    for row in range(1, len(df_accuracy_no_zero)+1):  # Start at 1 to skip header
        row_format = format_grey if row % 2 == 0 else format_white
        worksheet.set_row(row, 20, row_format)
    worksheet.set_column('A:A', 40)  # Set width for the first column
    worksheet.set_column('B:B', 20)  # Set width for the second column
    worksheet.set_column('C:C', 20)  # Set width for the third column

In [ ]:
## Operations that have accuracy between 0.49 and 0.51 that are among the non-zero features to understand why

# Select those that have accuracy between 0.49 and 0.51 among the non-zero
df_selected = df_accuracy_no_zero[(df_accuracy_no_zero['Accuracy'] > 0.49) & (df_accuracy_no_zero['Accuracy'] < 0.51)]

# Load df_ops_nonzero
df_ops_nonzero = pd.read_csv(dir_zero+'df_ops_nonzero.csv')

# extract from there the df_selected
df_ops_nonzero_selected = df_ops_nonzero[df_ops_nonzero['Name'].isin(df_selected['Operation'])]

In [ ]:
# Plot histogram of accuracy
data = df_zero_accuracy['Accuracy']
fig, ax = plt.subplots(figsize=(10, 6))  # 800x600 pixels (assuming 100 dpi)

# Histogram
n, bins, patches = ax.hist(data, bins=60, color='silver', edgecolor='grey', linewidth=1.2)

# Vertical line at threshold
ax.axvline(x=threshold_high, color='grey', linewidth=1.5, linestyle='--')

# Axis labels
ax.set_xlabel('Accuracy', fontsize=24, color='black')
ax.set_ylabel('Number of features', fontsize=24, color='black')

# Tick labels
ax.tick_params(axis='x', labelsize=24, colors='black')
ax.tick_params(axis='y', labelsize=24, colors='black')

# Grid lines
ax.xaxis.grid(True, color='whitesmoke')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.grid()
# no left y axis
ax.spines['left'].set_visible(True)
plt.title('Accuracy of zero features', fontsize=24, color='black')

# Optional: tighten layout
plt.tight_layout()
# Save in svg format
fig.savefig(dir_hctsa+'figures/histogram_accuracy_zero_h.svg', format='svg', dpi=300)


In [ ]:
# look how many good features are in the ties_column
df_good_hctsa_ties=df_good_hctsa[df_good_hctsa.columns.intersection(tie_columns)]
# Count how many unique values are in each column
df_good_hctsa_ties_counts = df_good_hctsa_ties.nunique() # number of unique values in each column